# Week 1 Lab 3 — Stella Oiro's Digital Career Twin

A Gradio chatbot that answers questions about Stella's career, powered by GPT-4o-mini.  
It adds a second-pass **evaluator** (Gemini or GPT-4o-mini) that scores every reply and forces a rerun if the quality falls short — so visitors always get a polished, accurate response.

**What this notebook builds, step by step:**
1. Load a LinkedIn PDF + personal summary as grounding context
2. Craft a system prompt so the model speaks as Stella
3. Build a structured evaluator that verdicts each reply
4. Add a `rerun` fallback that injects rejection feedback into the next attempt
5. Launch the full pipeline as a shareable Gradio chat interface

In [ ]:
# Core dependencies: OpenAI client, PDF reader, Gradio UI, and env loader
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [ ]:
from pathlib import Path

# Resolve paths to the LinkedIn PDF and personal summary — both live in me/ next to this notebook
_HERE = Path.cwd()
_ME = _HERE / "me"
_LINKEDIN_PDF = _ME / "linkedin.pdf"
_SUMMARY_TXT = _ME / "summary.txt"

In [ ]:
# Load API keys from .env and initialise the OpenAI client
load_dotenv(override=True)
openai = OpenAI()

In [ ]:
# Extract raw text from every page of the LinkedIn PDF to use as career context
reader = PdfReader(str(_LINKEDIN_PDF))
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [ ]:
# Sanity-check: confirm the PDF parsed correctly before wiring it into the prompt
print(linkedin)

In [ ]:
# Load the hand-written personal summary — a concise bio that supplements the LinkedIn data
with open(_SUMMARY_TXT, "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
# The name used throughout all prompts — update this to personalise the twin for someone else
name = "Stella Oiro"

In [ ]:
# Build the system prompt: instructs the model to stay in character using the LinkedIn + summary context
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [ ]:
# Inspect the assembled system prompt to verify context is injected correctly before proceeding
system_prompt

In [ ]:
# Basic chat function — no evaluator yet; used to verify the persona works before adding quality control
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [ ]:
# Quick smoke-test: launch the bare chat UI to confirm the persona responds correctly
gr.ChatInterface(chat, type="messages").launch()

In [ ]:
# Structured output schema: the evaluator returns a boolean verdict plus human-readable feedback
from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [ ]:
# Evaluator system prompt — the judge sees the same LinkedIn + summary context as the twin
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [ ]:
# Assembles the per-turn evaluator message: full conversation history + latest reply to score
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [ ]:
# Use Gemini as the evaluator if a Google key is available, otherwise fall back to GPT-4o-mini
import os

_gkey = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
if _gkey:
    gemini = OpenAI(
        api_key=_gkey,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )
    _EVAL_MODEL = "gemini-2.5-flash"
    _eval_client = gemini
else:
    _eval_client = openai
    _EVAL_MODEL = "gpt-4o-mini"
    print("GOOGLE_API_KEY not set — evaluator uses OpenAI gpt-4o-mini")

In [ ]:
# Runs the evaluator and returns a structured Evaluation object (verdict + feedback string)
def evaluate(reply, message, history) -> Evaluation:
    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = _eval_client.beta.chat.completions.parse(
        model=_EVAL_MODEL, messages=messages, response_format=Evaluation
    )
    return response.choices[0].message.parsed

In [ ]:
# Sample test query — generates a reply we can feed to the evaluator in the next cell
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [ ]:
# Inspect the raw reply before passing it to the evaluator
reply

In [ ]:
# Run the evaluator on the sample reply — expect is_acceptable=True for a straightforward honest answer
evaluate(reply, "do you hold a patent?", messages[:1])

In [ ]:
# Rerun: if a reply is rejected, inject the evaluator's feedback so the model can self-correct
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [ ]:
# Full pipeline: generate reply → evaluate → rerun with feedback if rejected; this is the production version
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply = response.choices[0].message.content

    evaluation = evaluate(reply, message, history)

    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)
    return reply

In [ ]:
# Find a free port and launch the full evaluator-backed chat as a shareable Gradio app
import os
import socket

def _pick_free_port(start: int, span: int = 40) -> int:
    """First bindable port in [start, start + span). Avoids stuck 7860 from a prior run."""
    for port in range(start, start + span):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(("0.0.0.0", port))
            except OSError:
                continue
            return port
    raise OSError(f"No free port in {start}–{start + span - 1}")


_start = int(os.getenv("PORT", os.getenv("GRADIO_SERVER_PORT", "7860")))
_port = _pick_free_port(_start)
print(f"Gradio using port {_port} (preferred start {_start})")

gr.ChatInterface(chat, type="messages").launch(
    share=True,
    server_name="0.0.0.0",
    server_port=_port,
)